In [ ]:
#%% Importing libraries
import transformers
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
import json
import re
import pandas as pd
import random
from tqdm import tqdm


/Users/karthik1705/Desktop/Projects/Sabre 2025/LLM Attribute Normalization/distiLLM/llm/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


# Prompt Engineer Gemini
# Prompt Engineer LLAMA


In [ ]:
from huggingface_hub import login
login(token='')

In [ ]:
#%% Load and prepare data
dataset_raw = pd.read_csv('Data/Product_Normalization_GRI.csv')
normalized_product_attributes = pd.read_excel('Data/Normalized_product_attribute_name.xlsx', sheet_name='Normalized Product Attributes')

#dataset_sample = pd.read_excel('sampled_descriptions_finetuning_llama_test.xlsx')

In [ ]:
len(dataset_raw['Room Description'].unique()), len(dataset_raw_expanded['Room Description Expanded'].unique())

(31237, 31237)

In [ ]:
#%% Loading Preprocessed (Expanded) Dataset
dataset_raw_expanded = pd.read_csv('Data/Product_Normalization_GRI_Expanded.csv')

In [ ]:
dataset_raw_expanded.head()

,Unnamed: 0,Room Description,Guest Room Info,Room Description Expanded
0,0,FLEXIBLE RATE|2QUEEN HEARING ACCESSIBLE STUDIO...,Accessible Room,FLEXIBLE RATE|2QUEEN HEARING ACCESSIBLE STUDIO...
1,1,2X POINTS PACKAGE|2QUEEN HEARING ACCESSIBLE ST...,Accessible Room,2X POINTS PACKAGE|2QUEEN HEARING ACCESSIBLE ST...
2,2,PARK AND STAY AAA|2QUEEN HEARING ACCESSIBLE ST...,Accessible Room,PARK AND STAY AAA|2QUEEN HEARING ACCESSIBLE ST...
3,3,PARK AND STAY AARP|2QUEEN HEARING ACCESSIBLE S...,Accessible Room,PARK AND STAY AARP|2QUEEN HEARING ACCESSIBLE S...
4,4,PARK AND GO|2QUEEN HEARING ACCESSIBLE STUDIO S...,Accessible Room,PARK AND GO|2QUEEN HEARING ACCESSIBLE STUDIO S...


In [ ]:
dataset_raw_expanded['Guest Room Info'].unique()

array(['Accessible Room', 'Suite', 'Executive/Club Suite', 'Double Bed',
       'King Bedroom', 'Queen Bedroom', 'Penthouse', 'Studio Suite',
       'Twin Room', 'Family Room/Suite', 'Cottage', 'Loft', 'Guest Room',
       nan, 'Bungalow', 'Villa', 'Junior Suite', 'Executive/Club Room',
       'Classic Room', 'Comfort Room', 'Deluxe Room', 'Deluxe Suite',
       'Premier Room', 'Standard Room', 'Superior Room', 'Superior Suite',
       'Premier Suite', 'Luxury Room', 'Classic Suite',
       'Presidential Suite', 'Single Room', 'Studio Room', 'Cabana',
       'Apartment', 'Luxury Suite', 'Run of the House'], dtype=object)

In [ ]:
# Get room types
room_types = normalized_product_attributes['RoomType'].dropna().unique()
room_types_list = room_types.tolist()
room_types_str = ", ".join(room_types_list)

In [ ]:
dataset_raw_expanded['Room Description Expanded'].unique()

array(['FLEXIBLE RATE|2QUEEN HEARING ACCESSIBLE STUDIO SUITE NON SMOKING VISUAL FIRE ALARM/DOOR/PHONE ALERT',
       '2X POINTS PACKAGE|2QUEEN HEARING ACCESSIBLE STUDIO SUITE NON SMOKING VISUAL FIRE ALARM/DOOR/PHONE ALERT',
       'PARK AND STAY AAA|2QUEEN HEARING ACCESSIBLE STUDIO SUITE NON SMOKING VISUAL FIRE ALARM/DOOR/PHONE ALERT',
       ...,
       'BEST FLEXIBLE RATE|HOLIDAY INN EXPRESS ROOM WHEN YOU ARRIVE WE WILL DO OUR BEST TO MEET YOUR ROOM BED TYPE AND SMOKING PREFERENCES. RMS SUBJ',
       'GREATRATE DISCOUNTED STAYS. 2|RUN OF HOUSE STANDARD ROOM DOUBLE BED OR 2 SINGLES WOODEN FLOOR',
       'CWT INCLUDES WIFI USE OF|FITNESS AND WELLNESS AREA AND BREAKFAST FOR 1 SPECIALTY ROOM WHEN YOU ARRIVE AT THE HOTEL WE WILL DO OUR BEST TO MEET YOUR ROOM TYPE PREFERENCE THIS IS SUBJECT TO'],
      dtype=object)

In [ ]:
def sample_descriptions_by_label_old(dataset, n_samples_per_class, random_state=42):
    """
    Sample Room Descriptions stratified by their Guest Room Info labels and return both sampled and remaining data
    """
    # Get unique labels
    labels = dataset['Guest Room Info'].unique()
    
    # Initialize lists to store samples
    sampled_descriptions = []
    sampled_descriptions_expanded = []  # Added for expanded descriptions
    sampled_labels = []
    sampled_indices = []
    
    print("\nSampling Room Descriptions by Label:")
    print("===================================")
    
    for label in labels:
        # Get all descriptions for this label
        label_data = dataset[dataset['Guest Room Info'] == label]
        
        # Calculate how many samples we can take
        n_available = len(label_data)
        n_to_sample = min(n_samples_per_class, n_available)
        
        if n_to_sample > 0:
            # Sample descriptions for this label
            sampled_data = label_data.sample(
                n=n_to_sample, 
                random_state=random_state
            )
            
            # Add to our lists
            sampled_descriptions.extend(sampled_data['Room Description'].tolist())
            sampled_descriptions_expanded.extend(sampled_data['Room Description Expanded'].tolist())
            sampled_labels.extend([label] * n_to_sample)
            sampled_indices.extend(sampled_data.index.tolist())
            
            print(f"\n{label}:")
            print(f"- Sampled {n_to_sample} descriptions (out of {n_available} available)")
            print(f"- Example: {sampled_data['Room Description'].iloc[0]}")
    
    # Create DataFrame of samples with both description columns
    samples_df = pd.DataFrame({
        'Room Description': sampled_descriptions,
        'Room Description Expanded': sampled_descriptions_expanded,
        'Guest Room Info': sampled_labels
    })
    
    # Create DataFrame of remaining data (not sampled)
    remaining_df = dataset[~dataset.index.isin(sampled_indices)].reset_index(drop=True)
    
    # Shuffle the samples
    samples_df = samples_df.sample(
        frac=1, 
        random_state=random_state
    ).reset_index(drop=True)
    
    print(f"\nTotal samples: {len(samples_df)} descriptions")
    print(f"Remaining data: {len(remaining_df)} descriptions")
    print("\nDistribution of labels in sample:")
    print(samples_df['Guest Room Info'].value_counts())
    
    return samples_df, remaining_df



In [ ]:
def create_stratified_sample(dataset, n_samples_per_class):
    """
    Creates a stratified sample from deduplicated dataset
    """
    # First deduplicate the dataset
    deduplicated_data = dataset.drop_duplicates(subset=['Room Description']).reset_index(drop=True)
    print(f"Original rows: {len(dataset)}")
    print(f"After deduplication: {len(deduplicated_data)}")
    
    # Rest of the sampling process
    room_types = deduplicated_data['Guest Room Info'].unique()
    sample_rows = []
    sampled_indices = set()
    
    print(f"\nSampling {n_samples_per_class} descriptions per room type:")
    print("=" * 50)
    
    for room_type in room_types:
        room_type_data = deduplicated_data[deduplicated_data['Guest Room Info'] == room_type]
        available = len(room_type_data)
        
        if available < n_samples_per_class:
            print(f"\nWarning: Only {available} samples available for {room_type}")
            n_to_sample = available
        else:
            n_to_sample = n_samples_per_class
            
        sampled = room_type_data.sample(n=n_to_sample, random_state=42)
        sample_rows.append(sampled)
        sampled_indices.update(sampled.index)
        
        print(f"\n{room_type}:")
        print(f"- Sampled {n_to_sample} from {available} available")
    
    sample_df = pd.concat(sample_rows, axis=0).reset_index(drop=True)
    remaining_df = deduplicated_data[~deduplicated_data.index.isin(sampled_indices)].reset_index(drop=True)
    
    print(f"\nFinal counts:")
    print(f"Sample size: {len(sample_df)}")
    print(f"Remaining size: {len(remaining_df)}")
    
    return sample_df, remaining_df


In [ ]:
# Usage:
sample_df, remaining_df = create_stratified_sample(dataset_raw_expanded, n_samples_per_class=100)

Original rows: 35000
After deduplication: 31237

Sampling 100 descriptions per room type:

Accessible Room:
- Sampled 100 from 962 available

Suite:
- Sampled 100 from 987 available

Executive/Club Suite:
- Sampled 100 from 860 available

Double Bed:
- Sampled 100 from 815 available

King Bedroom:
- Sampled 100 from 921 available

Queen Bedroom:
- Sampled 100 from 813 available

Penthouse:
- Sampled 100 from 975 available

Studio Suite:
- Sampled 100 from 864 available

Twin Room:
- Sampled 100 from 738 available

Family Room/Suite:
- Sampled 100 from 965 available

Cottage:
- Sampled 100 from 1000 available

Loft:
- Sampled 100 from 963 available

Guest Room:
- Sampled 100 from 694 available


nan:
- Sampled 0 from 0 available

Bungalow:
- Sampled 100 from 977 available

Villa:
- Sampled 100 from 927 available

Junior Suite:
- Sampled 100 from 922 available

Executive/Club Room:
- Sampled 100 from 981 available

Classic Room:
- Sampled 100 from 907 available

Comfort Room:
- Sampled 1

In [ ]:
sampled_data = sample_df
remaining_data = remaining_df

In [ ]:

# Usage:
# sampled_data, remaining_data = sample_descriptions_by_label(
#    dataset_raw_expanded, 
#    n_samples_per_class=100
#)
sampled_data.to_csv('sampled_data_finetuned_bert.csv', index = False)
remaining_data.to_csv('remaining_data_finetuned_bert.csv', index = False)

In [ ]:
sampled_data.head()

,Unnamed: 0,Room Description,Guest Room Info,Room Description Expanded
0,350,DREAM AWAY|1 SOFA PARLOR MOBILITY ACCESS TUB N...,Accessible Room,DREAM AWAY|1 SOFA PARLOR MOBILITY ACCESS TUB N...
1,387,TRIPADVISOR PLUS PROMOTIONAL R|ACCES 2 QN BEDS...,Accessible Room,TRIPADVISOR PLUS PROMOTIONAL R|ACCES 2 QUEEN B...
2,357,BREAKFAST INCLUDED|1 KING BED HEARING ACCESSIB...,Accessible Room,BREAKFAST INCLUDED|1 KING BED HEARING ACCESSIB...
3,797,FLEXIBLE RATE|2QN MOBILITY/HEARING ACCESS RI S...,Accessible Room,FLEXIBLE RATE|2QN MOBILITY/HEARING ACCESS RI S...
4,283,CWT SMALLBIZ -24HR CXL LATE CK|2 DOUBLE MOBILI...,Accessible Room,CWT SMALLBIZ -24HR CXL LATE CK|2 DOUBLE MOBILI...


# Sample descriptions
n_samples_per_class = 100 # v1 -> 10, v2 -> 100
sampled_data = sample_descriptions_by_label(
    dataset_raw_expanded, 
    n_samples_per_class=n_samples_per_class
)

# Save samples (optional)
sampled_data
#.to_csv('sampled_descriptions_finetuning_llama_test.csv', index=False)

In [ ]:
sampled_data

NameError: name 'sampled_data' is not defined

In [ ]:
remaining_data

,Unnamed: 0,Room Description,Guest Room Info,Room Description Expanded
0,0,FLEXIBLE RATE|2QUEEN HEARING ACCESSIBLE STUDIO...,Accessible Room,FLEXIBLE RATE|2QUEEN HEARING ACCESSIBLE STUDIO...
1,1,2X POINTS PACKAGE|2QUEEN HEARING ACCESSIBLE ST...,Accessible Room,2X POINTS PACKAGE|2QUEEN HEARING ACCESSIBLE ST...
2,2,PARK AND STAY AAA|2QUEEN HEARING ACCESSIBLE ST...,Accessible Room,PARK AND STAY AAA|2QUEEN HEARING ACCESSIBLE ST...
3,3,PARK AND STAY AARP|2QUEEN HEARING ACCESSIBLE S...,Accessible Room,PARK AND STAY AARP|2QUEEN HEARING ACCESSIBLE S...
4,4,PARK AND GO|2QUEEN HEARING ACCESSIBLE STUDIO S...,Accessible Room,PARK AND GO|2QUEEN HEARING ACCESSIBLE STUDIO S...
...,...,...,...,...
27732,34984,TRAVEL LEADERS WORLDWIDE|HOTEL STANDARD WHEN Y...,Run of the House,TRAVEL LEADERS WORLDWIDE|HOTEL STANDARD WHEN Y...
27733,34987,TRAVEL LEADERS WORLDWIDE|HOTEL STANDARD ROOMV ...,Run of the House,TRAVEL LEADERS WORLDWIDE|HOTEL STANDARD ROOMV ...
27734,34989,A T AND T INC GOLD|STANDARD NONSMOKING ROOM WH...,Run of the House,A T AND T INC GOLD|STANDARD NONSMOKING ROOM WH...
27735,34992,BEST FLEXIBLE RATE|HOLIDAY INN EXPRESS ROOM WH...,Run of the House,BEST FLEXIBLE RATE|HOLIDAY INN EXPRESS ROOM WH...


**BERT-Base-Uncased**

In [ ]:
from transformers import AutoModelForSequenceClassification, AutoTokenizer, TrainingArguments, Trainer
from datasets import Dataset
import numpy as np
from sklearn.metrics import accuracy_score, classification_report
import torch
from sklearn.model_selection import train_test_split

In [ ]:

def train_and_evaluate_bert_base_uncased(train_data, test_data, output_dir="bert_base_uncased_checkpoint"):
    """Initial training and evaluation on split data"""
    # Get unique labels and create label mapping
    labels = train_data['Guest Room Info'].unique()
    label2id = {label: i for i, label in enumerate(labels)}
    id2label = {i: label for label, i in label2id.items()}
    num_labels = len(labels)
    
    print(f"Number of labels: {num_labels}")
    
    model_name = "bert-base-uncased"
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForSequenceClassification.from_pretrained(
        model_name,
        num_labels=num_labels,
        id2label=id2label,
        label2id=label2id
    )
    
    def prepare_dataset(data):
        texts = data['Room Description'].tolist()
        labels = [label2id[label] for label in data['Guest Room Info']]
        
        tokenized = tokenizer(
            texts,
            padding=True,
            truncation=True,
            max_length=128,
            return_tensors="pt"
        )
        
        dataset = Dataset.from_dict({
            'input_ids': tokenized['input_ids'],
            'attention_mask': tokenized['attention_mask'],
            'labels': labels
        })
        return dataset
    
    train_dataset = prepare_dataset(train_data)
    test_dataset = prepare_dataset(test_data)
    
    def compute_metrics(eval_pred):
        predictions, labels = eval_pred
        predictions = np.argmax(predictions, axis=1)
        return {'accuracy': accuracy_score(labels, predictions)}
    
    training_args = TrainingArguments(
        output_dir=output_dir,
        num_train_epochs=3,
        per_device_train_batch_size=16,
        evaluation_strategy="epoch",
        save_strategy="epoch",
        learning_rate=2e-5,
        load_best_model_at_end=True,
        metric_for_best_model="accuracy"
    )
    
    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=test_dataset,
        compute_metrics=compute_metrics,
        tokenizer=tokenizer
    )
    
    print("Starting initial training...")
    trainer.train()
    
    # Evaluate
    print("\nEvaluating on test set...")
    predictions = trainer.predict(test_dataset)
    pred_labels = np.argmax(predictions.predictions, axis=1)
    true_labels = predictions.label_ids
    
    pred_labels_text = [id2label[id] for id in pred_labels]
    true_labels_text = [id2label[id] for id in true_labels]
    
    print("\nClassification Report:")
    print(classification_report(true_labels_text, pred_labels_text))
    
    return model_name, label2id, id2label



In [ ]:
sampled_data

,Unnamed: 0,Room Description,Guest Room Info,Room Description Expanded
0,350,DREAM AWAY|1 SOFA PARLOR MOBILITY ACCESS TUB N...,Accessible Room,DREAM AWAY|1 SOFA PARLOR MOBILITY ACCESS TUB N...
1,387,TRIPADVISOR PLUS PROMOTIONAL R|ACCES 2 QN BEDS...,Accessible Room,TRIPADVISOR PLUS PROMOTIONAL R|ACCES 2 QUEEN B...
2,357,BREAKFAST INCLUDED|1 KING BED HEARING ACCESSIB...,Accessible Room,BREAKFAST INCLUDED|1 KING BED HEARING ACCESSIB...
3,797,FLEXIBLE RATE|2QN MOBILITY/HEARING ACCESS RI S...,Accessible Room,FLEXIBLE RATE|2QN MOBILITY/HEARING ACCESS RI S...
4,283,CWT SMALLBIZ -24HR CXL LATE CK|2 DOUBLE MOBILI...,Accessible Room,CWT SMALLBIZ -24HR CXL LATE CK|2 DOUBLE MOBILI...
...,...,...,...,...
3495,34933,STATE GOVERNMENT|1 KING OR 2 QUEEN WHEN YOU AR...,Run of the House,STATE GOVERNMENT|1 KING OR 2 QUEEN WHEN YOU AR...
3496,34300,WALMART STORES INCORPORATED|GOLD STANDARD NONS...,Run of the House,WALMART STORES INCORPORATED|GOLD STANDARD NONS...
3497,34188,BCD TRAVEL RATE-CONSORTIA|RUN OF HOUSE KING OR...,Run of the House,BCD TRAVEL RATE-CONSORTIA|RUN OF HOUSE KING OR...
3498,34864,EARLY BOOKING RATE ROOM ONLY|RUN OF HOUSE ROOM...,Run of the House,EARLY BOOKING RATE ROOM ONLY|RUN OF HOUSE ROOM...


In [ ]:
def train_final_model(full_sample_data, model_name, label2id, output_dir="final_bert_base_uncased"):
    """Train final model on all 350 or 3500 samples"""
    id2label = {i: label for label, i in label2id.items()}
    num_labels = len(label2id)
    
    print("\nTraining final model on all sample data...")
    
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForSequenceClassification.from_pretrained(
        model_name,
        num_labels=num_labels,
        id2label=id2label,
        label2id=label2id
    )
    
    # Prepare full dataset
    texts = full_sample_data['Room Description'].tolist()
    labels = [label2id[label] for label in full_sample_data['Guest Room Info']]
    
    tokenized = tokenizer(
        texts,
        padding=True,
        truncation=True,
        max_length=128,
        return_tensors="pt"
    )
    
    full_dataset = Dataset.from_dict({
        'input_ids': tokenized['input_ids'],
        'attention_mask': tokenized['attention_mask'],
        'labels': labels
    })
    
    training_args = TrainingArguments(
        output_dir=output_dir,
        num_train_epochs=5,  # More epochs for final training
        per_device_train_batch_size=16,
        learning_rate=2e-5,
        save_strategy="epoch"
    )
    
    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=full_dataset,
        tokenizer=tokenizer
    )
    
    trainer.train()
    trainer.save_model()
    tokenizer.save_pretrained(output_dir)
    
    return model, tokenizer


In [ ]:
def classify_remaining_data(model, tokenizer, label2id, remaining_data):
    """Classify the remaining 35k descriptions"""
    device = torch.device('mps' if torch.backends.mps.is_available() else 'cpu')
    model = model.to(device)
    
    id2label = {i: label for label, i in label2id.items()}
    results = []
    
    print("\nClassifying remaining descriptions...")
    for _, row in tqdm(remaining_data.iterrows(), total=len(remaining_data)):
        inputs = tokenizer(
            row['Room Description'],
            padding=True,
            truncation=True,
            max_length=128,
            return_tensors="pt"
        )
        inputs = {k: v.to(device) for k, v in inputs.items()}
        
        outputs = model(**inputs)
        predictions = torch.nn.functional.softmax(outputs.logits, dim=-1)
        predicted_id = torch.argmax(predictions, dim=-1).item()
        confidence = predictions[0][predicted_id].item()
        
        results.append({
            'Room Description': row['Room Description'],
            'Actual_Room_Type': row['Guest Room Info'],  # Added actual label
            'Predicted_Room_Type': id2label[predicted_id],
            'Confidence': confidence
        })
    
    results_df = pd.DataFrame(results)
    
    # Add accuracy column
    results_df['Correct'] = results_df['Actual_Room_Type'] == results_df['Predicted_Room_Type']
    
    # Print overall accuracy
    accuracy = results_df['Correct'].mean()
    print(f"\nOverall Accuracy: {accuracy:.2%}")
    
    return results_df

In [ ]:
sampled_data

,Room Description,Guest Room Info
0,THE DINNER PACKAGE INCLUDES A|100 USD PER NIGH...,Executive/Club Room
1,SPA AND STAY|SUPERIOR SUITE-1KING OR 2TWINS-CO...,Superior Suite
2,REFUNDABLE RATE|STANDARD KING 250SQFT.STUNNING...,Standard Room
3,JP MORGAN CHASE|DELUXE ROOM-1KING-CITY VIEW-TV...,Deluxe Room
4,TRIPADVISOR PLUS|BALCONY VIEW DOUBLE DOUBLE RO...,Double Bed
...,...,...
3495,UNLIMITED SINGLE GOLF EXPERIEN|OUR 1811 KING C...,Cottage
3496,15PCT OFF. BKFST ART SUITE|KING BED:120SQM:LOF...,Loft
3497,PREPAY NONREF BKFT|PREPAY NON-REFUNDABLE WITH ...,Guest Room
3498,ACCENTURE-BREAKFAST INCLUDED|SUPERIOR DOUBLE O...,Twin Room


In [ ]:
# Main execution
# 1. Split sample data and validate
train_data, test_data = train_test_split(
    sampled_data, 
    test_size=0.2, 
    stratify=sampled_data['Guest Room Info'],
    random_state=42
)

# Initial training and evaluation
model_name, label2id, id2label = train_and_evaluate_bert_base_uncased(train_data, test_data)

Number of labels: 35


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/Users/karthik1705/Desktop/Projects/Sabre 2025/LLM Attribute Normalization/distiLLM/llm/lib/python3.9/site-packages/transformers/training_args.py:1575: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(
/var/folders/4s/6q_4fv2n6g94291jrzmhdsch0000gn/T/ipykernel_3159/10981553.py:58: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning

Starting initial training...


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
wandb: Currently logged in as: karthik1705 (karthik1705-the-university-of-texas-at-austin). Use `wandb login --relogin` to force relogin
wandb: Using wandb-core as the SDK backend.  Please refer to https://wandb.me/wandb-core for more information.
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Di

/Users/karthik1705/Desktop/Projects/Sabre 2025/LLM Attribute Normalization/distiLLM/llm/lib/python3.9/site-packages/torch/utils/data/dataloader.py:682: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Epoch,Training Loss,Validation Loss,Accuracy
1,No log,2.892358,0.500000
2,No log,1.691890,0.855714
3,2.509300,1.348354,0.905714


/Users/karthik1705/Desktop/Projects/Sabre 2025/LLM Attribute Normalization/distiLLM/llm/lib/python3.9/site-packages/torch/utils/data/dataloader.py:682: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, then device pinned memory won't be used.
  warnings.warn(warn_msg)
/Users/karthik1705/Desktop/Projects/Sabre 2025/LLM Attribute Normalization/distiLLM/llm/lib/python3.9/site-packages/torch/utils/data/dataloader.py:682: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, then device pinned memory won't be used.
  warnings.warn(warn_msg)
/Users/karthik1705/Desktop/Projects/Sabre 2025/LLM Attribute Normalization/distiLLM/llm/lib/python3.9/site-packages/torch/utils/data/dataloader.py:682: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, then device pinned memory won't be used.
  warnings.warn(warn_msg)



Evaluating on test set...


/Users/karthik1705/Desktop/Projects/Sabre 2025/LLM Attribute Normalization/distiLLM/llm/lib/python3.9/site-packages/torch/utils/data/dataloader.py:682: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, then device pinned memory won't be used.
  warnings.warn(warn_msg)



Classification Report:
                      precision    recall  f1-score   support

     Accessible Room       0.84      0.80      0.82        20
           Apartment       1.00      1.00      1.00        20
            Bungalow       1.00      1.00      1.00        20
              Cabana       0.91      1.00      0.95        20
        Classic Room       0.87      1.00      0.93        20
       Classic Suite       1.00      0.85      0.92        20
        Comfort Room       1.00      1.00      1.00        20
             Cottage       1.00      1.00      1.00        20
         Deluxe Room       0.89      0.80      0.84        20
        Deluxe Suite       0.94      0.85      0.89        20
          Double Bed       0.95      0.95      0.95        20
 Executive/Club Room       0.81      0.65      0.72        20
Executive/Club Suite       0.77      1.00      0.87        20
   Family Room/Suite       1.00      0.95      0.97        20
          Guest Room       0.86      0.90    

In [ ]:
# 2. Train on full sample dataset
final_model, final_tokenizer = train_final_model(sampled_data, model_name, label2id)



Training final model on all sample data...


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/var/folders/4s/6q_4fv2n6g94291jrzmhdsch0000gn/T/ipykernel_3159/1263641235.py:42: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(
/Users/karthik1705/Desktop/Projects/Sabre 2025/LLM Attribute Normalization/distiLLM/llm/lib/python3.9/site-packages/torch/utils/data/dataloader.py:682: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Step,Training Loss
500,2.225100
1000,0.413200


/Users/karthik1705/Desktop/Projects/Sabre 2025/LLM Attribute Normalization/distiLLM/llm/lib/python3.9/site-packages/torch/utils/data/dataloader.py:682: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, then device pinned memory won't be used.
  warnings.warn(warn_msg)
/Users/karthik1705/Desktop/Projects/Sabre 2025/LLM Attribute Normalization/distiLLM/llm/lib/python3.9/site-packages/torch/utils/data/dataloader.py:682: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, then device pinned memory won't be used.
  warnings.warn(warn_msg)
/Users/karthik1705/Desktop/Projects/Sabre 2025/LLM Attribute Normalization/distiLLM/llm/lib/python3.9/site-packages/torch/utils/data/dataloader.py:682: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, then device pinned memory won't be used.
  warnings.warn(warn_msg)
/Users/karthik1705/Desktop/Projects/Sabre 2025/LLM Attribute Normalization/distiLLM/llm/lib/python3.9/s

In [ ]:
dataset_raw_expanded

,Unnamed: 0,Room Description,Guest Room Info,Room Description Expanded
0,0,FLEXIBLE RATE|2QUEEN HEARING ACCESSIBLE STUDIO...,Accessible Room,FLEXIBLE RATE|2QUEEN HEARING ACCESSIBLE STUDIO...
1,1,2X POINTS PACKAGE|2QUEEN HEARING ACCESSIBLE ST...,Accessible Room,2X POINTS PACKAGE|2QUEEN HEARING ACCESSIBLE ST...
2,2,PARK AND STAY AAA|2QUEEN HEARING ACCESSIBLE ST...,Accessible Room,PARK AND STAY AAA|2QUEEN HEARING ACCESSIBLE ST...
3,3,PARK AND STAY AARP|2QUEEN HEARING ACCESSIBLE S...,Accessible Room,PARK AND STAY AARP|2QUEEN HEARING ACCESSIBLE S...
4,4,PARK AND GO|2QUEEN HEARING ACCESSIBLE STUDIO S...,Accessible Room,PARK AND GO|2QUEEN HEARING ACCESSIBLE STUDIO S...
...,...,...,...,...
34995,34995,CWT INCLUDES WIFI USE OF|FITNESS AND WELLNESS ...,Run of the House,CWT INCLUDES WIFI USE OF|FITNESS AND WELLNESS ...
34996,34996,STAY LONGER SAVE|SPECIALTY ROOM WHEN YOU ARRIV...,Run of the House,STAY LONGER SAVE|SPECIALTY ROOM WHEN YOU ARRIV...
34997,34997,CWT|SPECIALTY ROOM WHEN YOU ARRIVE AT THE HOTE...,Run of the House,CWT|SPECIALTY ROOM WHEN YOU ARRIVE AT THE HOTE...
34998,34998,"BOOK NOW, PAY LATER|DELUXE ROOM AN UPGRADE FRO...",Run of the House,"BOOK NOW, PAY LATER|DELUXE ROOM AN UPGRADE FRO..."


dataset_raw_expanded[~dataset_raw_expanded.index.isin(sampled_data.index)]

In [ ]:
# 3. Classify remaining data
#remaining_data = remaining_data
results_df = classify_remaining_data(final_model, final_tokenizer, label2id, remaining_data)



Classifying remaining descriptions...


  0%|          | 0/27737 [00:00<?, ?it/s]huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
100%|██████████| 27737/27737 [08:54<00:00, 51.92it/s]



Overall Accuracy: 95.30%


In [ ]:
results_df.to_csv('classified_rooms_bert_base_uncased_v4.csv', index=False)

Using Expanded Descriptions

In [ ]:
dataset_raw_expanded

,Unnamed: 0,Room Description,Guest Room Info,Room Description Expanded
0,0,FLEXIBLE RATE|2QUEEN HEARING ACCESSIBLE STUDIO...,Accessible Room,FLEXIBLE RATE|2QUEEN HEARING ACCESSIBLE STUDIO...
1,1,2X POINTS PACKAGE|2QUEEN HEARING ACCESSIBLE ST...,Accessible Room,2X POINTS PACKAGE|2QUEEN HEARING ACCESSIBLE ST...
2,2,PARK AND STAY AAA|2QUEEN HEARING ACCESSIBLE ST...,Accessible Room,PARK AND STAY AAA|2QUEEN HEARING ACCESSIBLE ST...
3,3,PARK AND STAY AARP|2QUEEN HEARING ACCESSIBLE S...,Accessible Room,PARK AND STAY AARP|2QUEEN HEARING ACCESSIBLE S...
4,4,PARK AND GO|2QUEEN HEARING ACCESSIBLE STUDIO S...,Accessible Room,PARK AND GO|2QUEEN HEARING ACCESSIBLE STUDIO S...
...,...,...,...,...
34995,34995,CWT INCLUDES WIFI USE OF|FITNESS AND WELLNESS ...,Run of the House,CWT INCLUDES WIFI USE OF|FITNESS AND WELLNESS ...
34996,34996,STAY LONGER SAVE|SPECIALTY ROOM WHEN YOU ARRIV...,Run of the House,STAY LONGER SAVE|SPECIALTY ROOM WHEN YOU ARRIV...
34997,34997,CWT|SPECIALTY ROOM WHEN YOU ARRIVE AT THE HOTE...,Run of the House,CWT|SPECIALTY ROOM WHEN YOU ARRIVE AT THE HOTE...
34998,34998,"BOOK NOW, PAY LATER|DELUXE ROOM AN UPGRADE FRO...",Run of the House,"BOOK NOW, PAY LATER|DELUXE ROOM AN UPGRADE FRO..."


In [ ]:
def train_final_model(full_sample_data, model_name, label2id, output_dir="final_bert_base_uncased"):
    """Train final model on all 3500 samples"""
    id2label = {i: label for label, i in label2id.items()}
    num_labels = len(label2id)
    
    print("\nTraining final model on all sample data...")
    
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForSequenceClassification.from_pretrained(
        model_name,
        num_labels=num_labels,
        id2label=id2label,
        label2id=label2id
    )
    
    # Prepare full dataset
    texts = full_sample_data['Room Description Expanded'].tolist()
    labels = [label2id[label] for label in full_sample_data['Guest Room Info']]
    
    tokenized = tokenizer(
        texts,
        padding=True,
        truncation=True,
        max_length=128,
        return_tensors="pt"
    )
    
    full_dataset = Dataset.from_dict({
        'input_ids': tokenized['input_ids'],
        'attention_mask': tokenized['attention_mask'],
        'labels': labels
    })
    
    training_args = TrainingArguments(
        output_dir=output_dir,
        num_train_epochs=5,  # More epochs for final training
        per_device_train_batch_size=16,
        learning_rate=2e-5,
        save_strategy="epoch"
    )
    
    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=full_dataset,
        tokenizer=tokenizer
    )
    
    trainer.train()
    trainer.save_model()
    tokenizer.save_pretrained(output_dir)
    
    return model, tokenizer


In [ ]:
def classify_remaining_data(model, tokenizer, label2id, remaining_data):
    """Classify the remaining 35k descriptions"""
    device = torch.device('mps' if torch.backends.mps.is_available() else 'cpu')
    model = model.to(device)
    
    id2label = {i: label for label, i in label2id.items()}
    results = []
    
    print("\nClassifying remaining descriptions...")
    for _, row in tqdm(remaining_data.iterrows(), total=len(remaining_data)):
        inputs = tokenizer(
            row['Room Description Expanded'],
            padding=True,
            truncation=True,
            max_length=128,
            return_tensors="pt"
        )
        inputs = {k: v.to(device) for k, v in inputs.items()}
        
        outputs = model(**inputs)
        predictions = torch.nn.functional.softmax(outputs.logits, dim=-1)
        predicted_id = torch.argmax(predictions, dim=-1).item()
        confidence = predictions[0][predicted_id].item()
        
        results.append({
            'Room Description': row['Room Description Expanded'],
            'Actual_Room_Type': row['Guest Room Info'],  # Added actual label
            'Predicted_Room_Type': id2label[predicted_id],
            'Confidence': confidence
        })
    
    results_df = pd.DataFrame(results)
    
    # Add accuracy column
    results_df['Correct'] = results_df['Actual_Room_Type'] == results_df['Predicted_Room_Type']
    
    # Print overall accuracy
    accuracy = results_df['Correct'].mean()
    print(f"\nOverall Accuracy: {accuracy:.2%}")
    
    return results_df

In [ ]:
# For expanded data, training on full sample dataset
expanded_final_model, expanded_final_tokenizer = train_final_model(sampled_data, model_name, label2id)



Training final model on all sample data...


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/var/folders/4s/6q_4fv2n6g94291jrzmhdsch0000gn/T/ipykernel_3159/406571484.py:42: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(
/Users/karthik1705/Desktop/Projects/Sabre 2025/LLM Attribute Normalization/distiLLM/llm/lib/python3.9/site-packages/torch/utils/data/dataloader.py:682: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Step,Training Loss
500,2.024700
1000,0.326300


wandb-core(5471) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
wandb-core(5476) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
wandb-core(5482) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
wandb-core(5484) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
wandb-core(5485) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
wandb-core(5487) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
wandb-core(5488) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
wandb-core(5491) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
wandb-core(5493) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
wandb-core(5494) MallocStackLogging: can't turn off malloc stack logging because it was not

In [ ]:
# For expanded data, classify remaining data
expanded_results_df = classify_remaining_data(expanded_final_model, expanded_final_tokenizer, label2id, remaining_data)



Classifying remaining descriptions...


100%|██████████| 27737/27737 [08:34<00:00, 53.88it/s]


Overall Accuracy: 96.18%


In [ ]:
expanded_results_df.to_csv('classified_rooms_bert_base_uncased_expanded.csv', index=False)

In [ ]:
expanded_results_df[expanded_results_df['Correct'] == False].groupby(['Correct', 'Actual_Room_Type', 'Predicted_Room_Type']).size()

Correct  Actual_Room_Type  Predicted_Room_Type
False    Accessible Room   Apartment              1
                           Deluxe Room            8
                           Double Bed             1
                           King Bedroom           6
                           Loft                   1
                                                 ..
         Twin Room         Executive/Club Room    7
         Villa             Bungalow               3
                           Loft                   1
                           Luxury Room            3
                           Penthouse              3
Length: 204, dtype: int64

In [ ]:
# 1. Misclassifications
errors_df = expanded_results_df[expanded_results_df['Correct'] == False].copy()

# 2. Error Pattern Analysis
error_analysis = errors_df.groupby(['Actual_Room_Type', 'Predicted_Room_Type']).agg({
    'Room Description': 'count',
    'Confidence': 'mean'
}).sort_values('Room Description', ascending=False)

print("Most common misclassifications:")
print(error_analysis.head(10))

# 3. Find low confidence correct predictions
low_conf_correct = expanded_results_df[
    (expanded_results_df['Correct'] == True) & 
    (expanded_results_df['Confidence'] < 0.8)
].sort_values('Confidence')

# 4. Find high confidence mistakes
high_conf_mistakes = errors_df[
    errors_df['Confidence'] > 0.9
].sort_values('Confidence', ascending=False)

# 5. Confusion by room type
confusion_by_type = pd.crosstab(
    expanded_results_df['Actual_Room_Type'], 
    expanded_results_df['Predicted_Room_Type']
)

Most common misclassifications:
                                          Room Description  Confidence
Actual_Room_Type    Predicted_Room_Type                               
Studio Suite        Studio Room                        119    0.673445
Executive/Club Room Executive/Club Suite                44    0.692434
Deluxe Room         Guest Room                          28    0.731037
Studio Room         Studio Suite                        25    0.579670
Luxury Suite        Presidential Suite                  23    0.888348
Suite               Accessible Room                     20    0.788242
Deluxe Room         Deluxe Suite                        19    0.719261
Luxury Room         Luxury Suite                        19    0.573669
Run of the House    Standard Room                       15    0.707924
Suite               Executive/Club Suite                15    0.750105


In [ ]:
errors_df[(errors_df['Actual_Room_Type'] == 'Studio Suite') & (errors_df['Predicted_Room_Type'] == 'Studio Room')].sort_values('Confidence', ascending=False)

,Room Description,Actual_Room_Type,Predicted_Room_Type,Confidence,Correct
6234,"RITZ KIDS PACKAGE|RITZ KIDS NIGHT SAFARI ADVENTURES, INCLUDES SEE RATE RULES, STUDIO, STUDIO, 1 KING, HUANG PU RIVER VIEW, CITY VIEW",Studio Suite,Studio Room,0.826681,False
5790,BED AND BREAKFAST|OCEAN VIEW STUDIO KING BED FULL MARBLE BATHROOM BUILDING 32 33 35 AND 36,Studio Suite,Studio Room,0.823553,False
6095,"SPECIAL EVENT RT PRGM|SPECIAL EVENT, SEE RATE RULES, STUDIO, 1 KING, SOFA BED",Studio Suite,Studio Room,0.811468,False
6316,"REGULAR RATE|FLEXIBLE RATE, FOUNTAIN SQUARE, LARGER STUDIO, 1 KING, FOUNTAIN SQUARE VIEW",Studio Suite,Studio Room,0.811155,False
6096,"SPECIAL EVENT RT PRGM|SPECIAL EVENT, SEE RATE RULES, STUDIO, 2 DOUBLE, SOFA BED",Studio Suite,Studio Room,0.802481,False
...,...,...,...,...,...
5693,CWT PROGRAM -DREAM AWAY|2 QUEEN BEDS STUDIO SUITE NONSMOKING HDTV/FREE WI-FI/ REFRIGERATOR/ MICROWAVE,Studio Suite,Studio Room,0.452197,False
6061,FLEXIBLE RATE|STUDIO KING ROOM- 52-58 SQM- KING BED - BATH- WALK- IN SHOWER- FREE WIFI- AC,Studio Suite,Studio Room,0.447077,False
6252,VIRTUOSO|STUDIO KING 65 SQM WITH BALCONY FREE EARLY C I AND LATE COCOMP MINIBAR,Studio Suite,Studio Room,0.439492,False
5772,BOOK EARLY SAVE|STUDIO OCEANFRONT *50 SQM*KING *PRIVATE BALCONY *SHOWER TUB SEPARATE *COMPL WIFI,Studio Suite,Studio Room,0.438227,False


In [ ]:
pd.set_option('display.max_colwidth', None)
high_conf_mistakes.head(10)

,Room Description,Actual_Room_Type,Predicted_Room_Type,Confidence,Correct
9485,"ADVANCE PURCHASE|1 KING 2 SINGLE BEDS,FULL BRKFST,NSMK,LOFT FAMILY ROOM,FIREPLACE",Loft,Family Room/Suite,0.925175,False
1514,BETTER TOGETHER|BUNGALOW SUITE - 1100 SQ FT - OCEAN VIEW 1K - LIVING ROOM - KITCHEN - ADULT ONLY,Suite,Bungalow,0.922629,False
891,"STAY FOR BREAKFST PREM|STAY FOR BREAKFAST RATE, INCLUDES BREAKFAST, DELUXE SUITE, 1 BEDROOM SUITE, 1 KING, COURTYARD VIEW",Suite,Deluxe Suite,0.921844,False
1127,OUR BEST AVAILABLE RATE|BUNGALOW SUITE - 1100 SQ FT - OCEAN VIEW 1K - LIVING ROOM - KITCHEN - ADULT ONLY,Suite,Bungalow,0.919601,False
1140,THE LINKS GOLF PACKAGE|BUNGALOW SUITE - 1100 SQ FT - OCEAN VIEW 1K - LIVING ROOM - KITCHEN - ADULT ONLY,Suite,Bungalow,0.917760,False
17529,PALAZZO MOST FLEXIBLE RATE|PREMIUM KING SUITE 940 SQUARE FEET 1 KING BED,Premier Room,Premier Suite,0.916994,False
17417,FLEXIBLE RATE|1 KING PREMIUM SUITE HEARING ACCESSIBLE VISUAL FIREALARM/DOOR/PHONE ALERT-2 ROOM SUITE,Premier Room,Premier Suite,0.916436,False
18650,WELLNESS PACKAGE|SUPERIOR CTYD QUEEN ROOM. 400 SQFT. COURTYRD VIEW LARGE MARBLE BATH SEP SHOWER-TUB. COMP WIFI,Superior Room,Queen Bedroom,0.913629,False
18895,FLEXIBLE RATE|SUPERIOR TWIN ROOM WITH 2 DOUBLE BEDS,Superior Room,Twin Room,0.913418,False
865,"VIRTUOSO PACKAGES|VIRTUOSO PACKAGE, JUNIOR SUITE, 1 BEDROOM SUITE, 1 KING, COURTYARD VIEW",Suite,Junior Suite,0.913227,False


In [ ]:
confusion_by_type.to_csv('confusion_by_type_expanded_results_df.csv', index=True)

In [ ]:
low_conf_correct.head(10)

,Room Description,Actual_Room_Type,Predicted_Room_Type,Confidence,Correct
1266,BEST AVAILABLE RATE MASTER|MANOR SUITE -KING ROOM. TOWER VIEW. LIVING ROOM. POWDER ROOM,Suite,Suite,0.144798,True
13222,BED AND BRKFST* 2 DOUBLE BEDS|2 DOUBLE BEDS:WORK DESK WITHEXECUTIVE CHAIR:,Executive/Club Room,Executive/Club Room,0.161756,True
13400,BED AND BRKFST* 1 KING BED|1 KING BED:WORK DESK WITHEXECUTIVE CHAIR:,Executive/Club Room,Executive/Club Room,0.191611,True
9747,TRIPADVISOR PLUS-INCL CONT BRK|GUESTROOMS FEATURES ONE QUEEN BED,Guest Room,Guest Room,0.196393,True
9922,"REGULAR RATE|FLEXIBLE RATE, WONDERFUL ROOM, GUEST ROOM, 1 KING",Guest Room,Guest Room,0.197722,True
13337,EXECUTIVE CLUB PACKAGE|TOWER ROOM KING. CITY VIEW. 85-98 FL. MARBLE BATHROOM. RAIN SHOWER. WC. BATHROBES.,Executive/Club Room,Executive/Club Room,0.214707,True
8382,DAILY RATE|COTTAGE EXECUTIVE SUITE-1 KING-COTTAGE VIEW OPEN PLAN-FIREPLACE-GARDEN PATIO-89SQM 958SQFT,Cottage,Cottage,0.216734,True
10207,"AAA AMERICAN AUTO ASSN|AAA CAA RATE, MEMBERSHIP CARD REQUIRED, NEWLY RENOVATED COZY KING, GUEST ROOM, 1 KING",Guest Room,Guest Room,0.216788,True
3227,NON-REFUNDABLE SAVE 40PCT|DUMBO KING BED ROOM 300 SQ FT.,King Bedroom,King Bedroom,0.217650,True
13551,1000 BONUS POINTS NT INCLUDES|ROOM AND 1000 REWARDS CLUB BONUS POINTS PER CLUB 1 KING BED ROOM SAPPHIRE 40 SQ METRES 1 KING BED ROOM WITH INDIVIDUAL BALCONY ENJOY THE VIEW OF ROCK CLIFF. ENJOY,Executive/Club Room,Executive/Club Room,0.220477,True


**MiniLM-L12-H384-uncased**

In [ ]:
from transformers import AutoModelForSequenceClassification, AutoTokenizer, TrainingArguments, Trainer
from datasets import Dataset
import numpy as np
from sklearn.metrics import accuracy_score, classification_report
import torch
from sklearn.model_selection import train_test_split

In [ ]:

def train_and_evaluate_minilm(train_data, test_data, output_dir="minilm_checkpoint"):
    """Initial training and evaluation on split data"""
    # Get unique labels and create label mapping
    labels = train_data['Guest Room Info'].unique()
    label2id = {label: i for i, label in enumerate(labels)}
    id2label = {i: label for label, i in label2id.items()}
    num_labels = len(labels)
    
    print(f"Number of labels: {num_labels}")
    
    model_name = "microsoft/MiniLM-L12-H384-uncased"
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForSequenceClassification.from_pretrained(
        model_name,
        num_labels=num_labels,
        id2label=id2label,
        label2id=label2id
    )
    
    def prepare_dataset(data):
        texts = data['Room Description'].tolist()
        labels = [label2id[label] for label in data['Guest Room Info']]
        
        tokenized = tokenizer(
            texts,
            padding=True,
            truncation=True,
            max_length=128,
            return_tensors="pt"
        )
        
        dataset = Dataset.from_dict({
            'input_ids': tokenized['input_ids'],
            'attention_mask': tokenized['attention_mask'],
            'labels': labels
        })
        return dataset
    
    train_dataset = prepare_dataset(train_data)
    test_dataset = prepare_dataset(test_data)
    
    def compute_metrics(eval_pred):
        predictions, labels = eval_pred
        predictions = np.argmax(predictions, axis=1)
        return {'accuracy': accuracy_score(labels, predictions)}
    
    training_args = TrainingArguments(
        output_dir=output_dir,
        num_train_epochs=3,
        per_device_train_batch_size=16,
        evaluation_strategy="epoch",
        save_strategy="epoch",
        learning_rate=2e-5,
        load_best_model_at_end=True,
        metric_for_best_model="accuracy"
    )
    
    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=test_dataset,
        compute_metrics=compute_metrics,
        tokenizer=tokenizer
    )
    
    print("Starting initial training...")
    trainer.train()
    
    # Evaluate
    print("\nEvaluating on test set...")
    predictions = trainer.predict(test_dataset)
    pred_labels = np.argmax(predictions.predictions, axis=1)
    true_labels = predictions.label_ids
    
    pred_labels_text = [id2label[id] for id in pred_labels]
    true_labels_text = [id2label[id] for id in true_labels]
    
    print("\nClassification Report:")
    print(classification_report(true_labels_text, pred_labels_text))
    
    return model_name, label2id, id2label



In [ ]:
def train_final_model_minilm(full_sample_data, model_name, label2id, output_dir="final_minilm"):
    """Train final model on all 350 samples"""
    id2label = {i: label for label, i in label2id.items()}
    num_labels = len(label2id)
    
    print("\nTraining final model on all sample data...")
    
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForSequenceClassification.from_pretrained(
        model_name,
        num_labels=num_labels,
        id2label=id2label,
        label2id=label2id
    )
    
    # Prepare full dataset
    texts = full_sample_data['Room Description'].tolist()
    labels = [label2id[label] for label in full_sample_data['Guest Room Info']]
    
    tokenized = tokenizer(
        texts,
        padding=True,
        truncation=True,
        max_length=128,
        return_tensors="pt"
    )
    
    full_dataset = Dataset.from_dict({
        'input_ids': tokenized['input_ids'],
        'attention_mask': tokenized['attention_mask'],
        'labels': labels
    })
    
    training_args = TrainingArguments(
        output_dir=output_dir,
        num_train_epochs=5,  # More epochs for final training
        per_device_train_batch_size=16,
        learning_rate=2e-5,
        save_strategy="epoch"
    )
    
    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=full_dataset,
        tokenizer=tokenizer
    )
    
    trainer.train()
    trainer.save_model()
    tokenizer.save_pretrained(output_dir)
    
    return model, tokenizer




In [ ]:
def classify_remaining_data_minilm(model, tokenizer, label2id, remaining_data):
    """Classify the remaining 35k descriptions"""
    device = torch.device('mps' if torch.backends.mps.is_available() else 'cpu')
    model = model.to(device)
    
    id2label = {i: label for label, i in label2id.items()}
    results = []
    
    print("\nClassifying remaining descriptions...")
    for _, row in tqdm(remaining_data.iterrows(), total=len(remaining_data)):
        inputs = tokenizer(
            row['Room Description'],
            padding=True,
            truncation=True,
            max_length=128,
            return_tensors="pt"
        )
        inputs = {k: v.to(device) for k, v in inputs.items()}
        
        outputs = model(**inputs)
        predictions = torch.nn.functional.softmax(outputs.logits, dim=-1)
        predicted_id = torch.argmax(predictions, dim=-1).item()
        confidence = predictions[0][predicted_id].item()
        
        results.append({
            'Room Description': row['Room Description'],
            'Actual_Room_Type': row['Guest Room Info'],  # Added actual label
            'Predicted_Room_Type': id2label[predicted_id],
            'Confidence': confidence
        })
    
    results_df = pd.DataFrame(results)
    
    # Add accuracy column
    results_df['Correct'] = results_df['Actual_Room_Type'] == results_df['Predicted_Room_Type']
    
    # Print overall accuracy
    accuracy = results_df['Correct'].mean()
    print(f"\nOverall Accuracy: {accuracy:.2%}")
    
    return results_df

In [ ]:
# Main execution
# 1. Split sample data and validate
train_data, test_data = train_test_split(
    sampled_data, 
    test_size=0.2, 
    stratify=sampled_data['Guest Room Info'],
    random_state=42
)

# Initial training and evaluation
model_name, label2id, id2label = train_and_evaluate_minilm(train_data, test_data)

Number of labels: 35


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at microsoft/MiniLM-L12-H384-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/Users/karthik1705/Desktop/Projects/Sabre 2025/LLM Attribute Normalization/distiLLM/llm/lib/python3.9/site-packages/transformers/training_args.py:1575: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(
/var/folders/4s/6q_4fv2n6g94291jrzmhdsch0000gn/T/ipykernel_26682/248026038.py:58: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Starting initial training...


/Users/karthik1705/Desktop/Projects/Sabre 2025/LLM Attribute Normalization/distiLLM/llm/lib/python3.9/site-packages/torch/utils/data/dataloader.py:682: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Epoch,Training Loss,Validation Loss,Accuracy
1,No log,3.413036,0.068571
2,No log,3.232183,0.130000
3,3.352500,3.174551,0.152857


/Users/karthik1705/Desktop/Projects/Sabre 2025/LLM Attribute Normalization/distiLLM/llm/lib/python3.9/site-packages/torch/utils/data/dataloader.py:682: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, then device pinned memory won't be used.
  warnings.warn(warn_msg)
/Users/karthik1705/Desktop/Projects/Sabre 2025/LLM Attribute Normalization/distiLLM/llm/lib/python3.9/site-packages/torch/utils/data/dataloader.py:682: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, then device pinned memory won't be used.
  warnings.warn(warn_msg)
/Users/karthik1705/Desktop/Projects/Sabre 2025/LLM Attribute Normalization/distiLLM/llm/lib/python3.9/site-packages/torch/utils/data/dataloader.py:682: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, then device pinned memory won't be used.
  warnings.warn(warn_msg)



Evaluating on test set...


/Users/karthik1705/Desktop/Projects/Sabre 2025/LLM Attribute Normalization/distiLLM/llm/lib/python3.9/site-packages/torch/utils/data/dataloader.py:682: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, then device pinned memory won't be used.
  warnings.warn(warn_msg)



Classification Report:
                      precision    recall  f1-score   support

     Accessible Room       0.00      0.00      0.00        20
           Apartment       0.00      0.00      0.00        20
            Bungalow       0.14      1.00      0.25        20
              Cabana       0.00      0.00      0.00        20
        Classic Room       0.00      0.00      0.00        20
       Classic Suite       0.12      0.20      0.15        20
        Comfort Room       0.00      0.00      0.00        20
             Cottage       0.00      0.00      0.00        20
         Deluxe Room       0.88      0.70      0.78        20
        Deluxe Suite       0.00      0.00      0.00        20
          Double Bed       0.00      0.00      0.00        20
 Executive/Club Room       0.00      0.00      0.00        20
Executive/Club Suite       0.27      0.30      0.29        20
   Family Room/Suite       0.00      0.00      0.00        20
          Guest Room       0.23      0.75    

/Users/karthik1705/Desktop/Projects/Sabre 2025/LLM Attribute Normalization/distiLLM/llm/lib/python3.9/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/Users/karthik1705/Desktop/Projects/Sabre 2025/LLM Attribute Normalization/distiLLM/llm/lib/python3.9/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/Users/karthik1705/Desktop/Projects/Sabre 2025/LLM Attribute Normalization/distiLLM/llm/lib/python3.9/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and bein

In [ ]:

# 2. Train on full sample dataset
final_model, final_tokenizer = train_final_model_minilm(sampled_data, model_name, label2id)



Training final model on all sample data...


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/var/folders/4s/6q_4fv2n6g94291jrzmhdsch0000gn/T/ipykernel_26682/3187214905.py:42: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(
/Users/karthik1705/Desktop/Projects/Sabre 2025/LLM Attribute Normalization/distiLLM/llm/lib/python3.9/site-packages/torch/utils/data/dataloader.py:682: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Step,Training Loss
500,2.065200
1000,0.339400


/Users/karthik1705/Desktop/Projects/Sabre 2025/LLM Attribute Normalization/distiLLM/llm/lib/python3.9/site-packages/torch/utils/data/dataloader.py:682: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, then device pinned memory won't be used.
  warnings.warn(warn_msg)
/Users/karthik1705/Desktop/Projects/Sabre 2025/LLM Attribute Normalization/distiLLM/llm/lib/python3.9/site-packages/torch/utils/data/dataloader.py:682: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, then device pinned memory won't be used.
  warnings.warn(warn_msg)
/Users/karthik1705/Desktop/Projects/Sabre 2025/LLM Attribute Normalization/distiLLM/llm/lib/python3.9/site-packages/torch/utils/data/dataloader.py:682: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, then device pinned memory won't be used.
  warnings.warn(warn_msg)
/Users/karthik1705/Desktop/Projects/Sabre 2025/LLM Attribute Normalization/distiLLM/llm/lib/python3.9/s

In [ ]:
final_model

BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSdpaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e

In [ ]:

# 3. Classify remaining data
remaining_data = dataset_raw[~dataset_raw.index.isin(sampled_data.index)]
results_df = classify_remaining_data_minilm(final_model, final_tokenizer, label2id, remaining_data)


Classifying remaining descriptions...


 89%|████████▉ | 28174/31500 [36:32<04:18, 12.85it/s]  


KeyboardInterrupt: 

In [ ]:
# Save results
results_df.to_csv('classified_rooms_minilm_v2.csv', index=False)
print("\nClassification complete! Results saved to 'classified_rooms_minilm_v2.csv'")


Classification complete! Results saved to 'classified_rooms_minilm_v2.csv'
